In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Convert timestamp columns to timezone-aware datetimes in UTC+8 (Asia/Shanghai)
# Store new columns to avoid losing original epoch seconds
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_trace['ts']  = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_log['ts']    = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
df_err['ts']    = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Number of rows per file
row_counts = pd.Series({
    'metric.csv': df_metric.shape[0],
    'trace.csv': df_trace.shape[0],
    'log.csv': df_log.shape[0],
    'error_logs.csv': df_err.shape[0]
}, name='row_count')

# 2) Unique cmdb_id values (sample up to 50) and counts per file
def cmdb_summary(df, file_label, limit=50):
    uniq = pd.Series(df['cmdb_id'].dropna().unique())
    sample = ','.join(uniq.astype(str).iloc[:limit].tolist())
    return pd.Series({
        'file': file_label,
        'unique_cmdb_count': int(uniq.shape[0]),
        'sample_cmdb_ids': sample
    })

cmdb_summaries = pd.DataFrame([
    cmdb_summary(df_metric, 'metric.csv'),
    cmdb_summary(df_trace, 'trace.csv'),
    cmdb_summary(df_log, 'log.csv'),
    cmdb_summary(df_err, 'error_logs.csv')
]).set_index('file')

# 3) metric.csv: unique kpi_name values (limit 50) and aggregation by (cmdb_id, kpi_name)
metric_kpis = pd.Series(df_metric['kpi_name'].dropna().unique()[:50], name='sample_kpi_names')

# Aggregate using full series for quantile calculations
metric_agg_full = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].agg(
    count='count',
    p95=lambda s: s.quantile(0.95),
    p5 =lambda s: s.quantile(0.05)
)

metric_agg_full[['p95','p5']] = metric_agg_full[['p95','p5']].astype(float)
metric_agg_top20 = metric_agg_full.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
# Round for compact display
metric_agg_top20[['p95','p5']] = metric_agg_top20[['p95','p5']].round(6)

# 4) trace.csv: unique trace_name values (limit 50) and aggregation by (cmdb_id, trace_name)
trace_names = pd.Series(df_trace['trace_name'].dropna().unique()[:50], name='sample_trace_names')

trace_agg_full = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].agg(
    count='count',
    p95=lambda s: s.quantile(0.95)
)
trace_agg_full['p95'] = trace_agg_full['p95'].astype(float)
trace_agg_top20 = trace_agg_full.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
trace_agg_top20['p95'] = trace_agg_top20['p95'].round(6)

# 5) log.csv: unique log_name values (limit 50) and aggregation by (cmdb_id, log_name)
log_names = pd.Series(df_log['log_name'].dropna().unique()[:50], name='sample_log_names')

log_agg_full = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].agg(
    count='count',
    p95=lambda s: s.quantile(0.95)
)
log_agg_full['p95'] = log_agg_full['p95'].astype(float)
log_agg_top20 = log_agg_full.sort_values('p95', ascending=False).head(20).reset_index(drop=True)
log_agg_top20['p95'] = log_agg_top20['p95'].round(6)

# 6) error_logs.csv: number of rows, up to 20 distinct cmdb_id values, earliest and latest timestamp
error_unique_cmdb = pd.Series(df_err['cmdb_id'].dropna().unique()[:20])
error_earliest = df_err['ts'].min()
error_latest = df_err['ts'].max()

error_summary = pd.Series({
    'error_rows': int(df_err.shape[0]),
    'distinct_cmdb_sample_up_to_20': ','.join(error_unique_cmdb.astype(str).tolist()),
    'earliest_ts_Asia_Shanghai': error_earliest,
    'latest_ts_Asia_Shanghai': error_latest
})

# Final required displayed outputs (compact)
row_counts, cmdb_summaries, metric_kpis, metric_agg_top20[['cmdb_id','kpi_name','count','p95','p5']], trace_names, trace_agg_top20[['cmdb_id','trace_name','count','p95']], log_names, log_agg_top20[['cmdb_id','log_name','count','p95']], error_summary

```
Out[1]:
```
# Summarize results in plain English using existing kernel variables
summary = (
    f"Dataset sizes: metric.csv={int(row_counts['metric.csv'])} rows, "
    f"trace.csv={int(row_counts['trace.csv'])} rows, "
    f"log.csv={int(row_counts['log.csv'])} rows, "
    f"error_logs.csv={int(row_counts['error_logs.csv'])} rows.\n\n"
    f"Unique component counts (sample shown): metric={int(cmdb_summaries.loc['metric.csv','unique_cmdb_count'])} "
    f"(e.g. {cmdb_summaries.loc['metric.csv','sample_cmdb_ids'].split(',')[0:5]}), "
    f"trace={int(cmdb_summaries.loc['trace.csv','unique_cmdb_count'])}, "
    f"log={int(cmdb_summaries.loc['log.csv','unique_cmdb_count'])}, "
    f"error_logs={int(cmdb_summaries.loc['error_logs.csv','unique_cmdb_count'])} "
    f"(sample: {cmdb_summaries.loc['error_logs.csv','sample_cmdb_ids']}).\n\n"
    "Top metric (by P95) examples (cmdb_id, kpi_name, p95):\n"
    + "\n".join(
        f"  {i+1}. {row['cmdb_id']} | {row['kpi_name']} | p95={row['p95']}"
        for i, row in metric_agg_top20.head(3).iterrows()
    )
    + "\n\nTop trace (by P95) examples (cmdb_id, trace_name, p95):\n"
    + "\n".join(
        f"  {i+1}. {row['cmdb_id']} | {row['trace_name']} | p95={row['p95']}"
        for i, row in trace_agg_top20.head(3).iterrows()
    )
    + "\n\nTop logs (by P95) examples (cmdb_id, log_name, p95):\n"
    + "\n".join(
        f"  {i+1}. {row['cmdb_id']} | {row['log_name']} | p95={row['p95']}"
        for i, row in log_agg_top20.head(3).iterrows()
    )
    + f"\n\nError logs: {int(error_summary['error_rows'])} rows; sample cmdb_ids: {error_summary['distinct_cmdb_sample_up_to_20']}. "
    f"Earliest error timestamp (Asia/Shanghai) = {error_summary['earliest_ts_Asia_Shanghai']}, "
    f"latest = {error_summary['latest_ts_Asia_Shanghai']}."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(metric.csv        147520
trace.csv          25008
log.csv             1560
error_logs.csv       319
Name: row_count, dtype: int64,                 unique_cmdb_count                                    sample_cmdb_ids
file                                                                                
metric.csv                     59  adservice,adservice-0,adservice-1,adservice-2,...
trace.csv                      40  adservice-0,adservice-1,adservice-2,adservice2...
log.csv                        31  adservice-0,adservice-1,adservice-2,cartservic...
error_logs.csv                  3                   frontend-1,frontend-0,frontend-2, 0                                        app.grpc.count
1                                          app.grpc.mrt
2                                           app.grpc.rr
3                                           app.grpc.sr
4                                        app.http.count
                            ...                        
45    runtime.java_lang_GarbageCollector_LastGcInfo_...
46    runtime.java_lang_GarbageCollector_LastGcInfo_...
47    runtime.java_lang_GarbageCollector_LastGcInfo_...
48    runtime.java_lang_GarbageCollector_LastGcInfo_...
49    runtime.java_lang_GarbageCollector_LastGcInfo_...
Name: sample_kpi_names, Length: 50, dtype: object,        cmdb_id                                           kpi_name  count           p95            p5
0   adservice2                runtime.java_lang_Runtime_StartTime     30  1.647789e+12  1.647789e+12
1    adservice                runtime.java_lang_Runtime_StartTime     30  1.647523e+12  1.647523e+12
2    adservice   runtime.java_lang_OperatingSystem_ProcessCpuTime     30  9.420655e+11  9.382493e+11
3   adservice2   runtime.java_lang_OperatingSystem_ProcessCpuTime     30  1.367404e+11  1.310157e+11
4    adservice   runtime.java_lang_Threading_CurrentThreadCpuTime     30  9.097850e+10  9.049630e+10
5    adservice  runtime.java_lang_Threading_CurrentThreadUserTime     30  8.849150e+10  8.802246e+10
6    adservice  runtime.java_lang_Threading_CurrentThreadAlloc...     30  7.212517e+10  7.174321e+10
7   adservice2   runtime.java_lang_Threading_CurrentThreadCpuTime     30  1.070184e+10  1.017195e+10
8   adservice2  runtime.java_lang_Threading_CurrentThreadUserTime     30  9.965917e+09  9.474250e+09
9       node-4                                  system.disk.total     30  7.616300e+09  7.616300e+09
10  adservice2  runtime.java_lang_Threading_CurrentThreadAlloc...     30  7.444690e+09  7.059730e+09
11      node-4                             system.fs.inodes.total     30  6.567638e+09  6.562516e+09
12      node-4                              system.fs.inodes.free     30  6.565049e+09  6.559927e+09
13      node-3                                  system.disk.total     30  5.867011e+09  5.867011e+09
14      node-2                                  system.disk.total     30  5.867011e+09  5.867011e+09
15      node-1                                  system.disk.total     30  5.867011e+09  5.867011e+09
16      node-4                                   system.disk.used     30  5.779297e+09  5.778016e+09
17      node-3                             system.fs.inodes.total     30  4.935860e+09  4.931316e+09
18      node-1                             system.fs.inodes.total     30  4.935536e+09  4.931712e+09
19      node-2                             system.fs.inodes.total     30  4.935489e+09  4.931712e+09, 0          trace.from_frontend-0.duration_mean
1           trace.from_frontend-0.duration_p95
2             trace.from_frontend-0.error_rate
3              trace.from_frontend-0.row_count
4          trace.from_frontend-1.duration_mean
                        ...                   
45         trace.to_cartservice-2.duration_p95
46           trace.to_cartservice-2.error_rate
47            trace.to_cartservice-2.row_count
48    trace.to_currencyservice-0.duration_mean
49     trace.to_currencyservice-0.duration_p95
Name: sample_trace_names, Length: 50, dtype: object,                      cmdb_id                                   trace_name  count     p95
0                frontend2-0                         trace.self.row_count     30  843.00
1   productcatalogservice2-0             trace.from_frontend2-0.row_count     30  408.95
2                frontend2-0  trace.to_productcatalogservice2-0.row_count     30  408.95
3                 frontend-0                         trace.self.row_count     30  311.65
4         currencyservice2-0             trace.from_frontend2-0.row_count     30  248.85
5                frontend2-0        trace.to_currencyservice2-0.row_count     30  248.85
6                 frontend-1                         trace.self.row_count     30  169.05
7                 frontend-2                         trace.self.row_count     30  159.00
8             cartservice2-0                         trace.self.row_count     30   88.00
9                frontend2-0                    trace.from_root.row_count     30   84.20
10            cartservice2-0             trace.from_frontend2-0.row_count     30   76.10
11               frontend2-0            trace.to_cartservice2-0.row_count     30   76.10
12               frontend2-0  trace.to_recommendationservice2-0.row_count     30   64.10
13  recommendationservice2-0                         trace.self.row_count     30   64.10
14  recommendationservice2-0             trace.from_frontend2-0.row_count     30   64.10
15               frontend2-0              trace.to_adservice2-0.row_count     30   51.00
16              adservice2-0             trace.from_frontend2-0.row_count     30   51.00
17   productcatalogservice-1              trace.from_frontend-0.row_count     30   46.55
18   productcatalogservice-2              trace.from_frontend-0.row_count     30   46.55
19                frontend-0   trace.to_productcatalogservice-1.row_count     30   46.55, 0    log.error_count
1      log.row_count
Name: sample_log_names, dtype: object,                     cmdb_id       log_name  count     p95
0                frontend-0  log.row_count     30  491.40
1               adservice-0  log.row_count     30  318.00
2   productcatalogservice-2  log.row_count     30  265.85
3   productcatalogservice-1  log.row_count     30  265.45
4   productcatalogservice-0  log.row_count     30  264.45
5                frontend-1  log.row_count     30  261.65
6                frontend-2  log.row_count     30  244.55
7             cartservice-0  log.row_count     30  195.95
8             cartservice-1  log.row_count     30  195.40
9             cartservice-2  log.row_count     30  189.55
10        currencyservice-1  log.row_count     30  179.65
11        currencyservice-2  log.row_count     30  179.30
12        currencyservice-0  log.row_count     30  174.55
13  recommendationservice-0  log.row_count     30   56.10
14  recommendationservice-2  log.row_count     30   55.20
15  recommendationservice-1  log.row_count     30   53.55
16        checkoutservice-0  log.row_count     13   50.00
17        shippingservice-1  log.row_count     29   36.00
18        checkoutservice-1  log.row_count     16   34.00
19        checkoutservice-2  log.row_count     15   34.00, error_rows                                                    319
distinct_cmdb_sample_up_to_20    frontend-1,frontend-0,frontend-2
earliest_ts_Asia_Shanghai               2022-03-21 07:00:00+08:00
latest_ts_Asia_Shanghai                 2022-03-21 07:29:55+08:00
dtype: object)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables if present; otherwise load
try:
    df_metric, tz
except NameError:
    df_metric = pd.read_csv('metric.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Define target cmdb_ids
targets = ['frontend-0','frontend-1','frontend-2','frontend2-0',
           'node-1','node-2','node-3','node-4','node-5','node-6']

# 2) Compute global thresholds (count, p95, p5) for (cmdb_id, kpi_name) using full series
metric_thresholds = (
    df_metric[df_metric['cmdb_id'].isin(targets)]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda s: float(s.quantile(0.95)),
         p5 =lambda s: float(s.quantile(0.05)))
)

# 3) Filter metric.csv to the incident window (inclusive)
start = pd.to_datetime('2022-03-21 07:00:00').tz_localize(tz)
end   = pd.to_datetime('2022-03-21 07:30:00').tz_localize(tz)

metric_window = df_metric[
    (df_metric['cmdb_id'].isin(targets)) &
    (df_metric['ts'] >= start) &
    (df_metric['ts'] <= end)
].copy()

# 4) Join metric_window with metric_thresholds on (cmdb_id,kpi_name)
mw_joined = metric_window.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left', suffixes=('','_global'))

# Per-row anomaly flag using global p95/p5
mw_joined['is_anomaly'] = (
    mw_joined.apply(lambda r: (pd.notnull(r['p95']) and r['value'] >= r['p95']) or
                              (pd.notnull(r['p5'])  and r['value'] <= r['p5']), axis=1)
)

# Aggregate per (cmdb_id,kpi_name) for window stats
grouped = mw_joined.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    count = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# earliest anomaly timestamp per group (if any)
earliest_anom = (
    mw_joined[mw_joined['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts':'earliest_anomaly_ts'})
)

# Merge earliest anomaly ts
result = grouped.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Ensure timezone for earliest_anomaly_ts (may be NaT)
result['earliest_anomaly_ts'] = pd.to_datetime(result['earliest_anomaly_ts']).dt.tz_convert(tz)

# 5) Return top 20 rows sorted by anomaly_count desc. If no anomalies, sort by deviation score desc.
total_anomalies = int(result['anomaly_count'].sum()) if not result.empty else 0

if total_anomalies > 0:
    result_top20 = result.sort_values(['anomaly_count','count'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    # compute deviation score = max(|max - p95|, |min - p5|)
    result['dev_score'] = result.apply(lambda r: max(
        abs(r['max_value_in_window'] - r['p95']) if pd.notnull(r['p95']) else 0,
        abs(r['min_value_in_window'] - r['p5'])  if pd.notnull(r['p5'])  else 0
    ), axis=1)
    result_top20 = result.sort_values('dev_score', ascending=False).head(20).reset_index(drop=True)
    result_top20 = result_top20.drop(columns=['dev_score'], errors='ignore')

# Compact display: requested columns only
result_top20_display = result_top20[['cmdb_id','kpi_name','count','p95','p5','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window']].head(20)

# Expose variables for later steps and display compact result
metric_thresholds, metric_window.shape, result_top20_display

```
Out[2]:
```
```python
summary = (
    "Summary of metric inspection for targets ['frontend-0','frontend-1','frontend-2',"
    "'frontend2-0','node-1','node-2','node-3','node-4','node-5','node-6']:\n\n"
    "- Global thresholds computed for 1012 (cmdb_id, kpi_name) combinations (metric_thresholds).\n"
    "- Filtered metric window (2022-03-21 07:00:00+08 to 2022-03-21 07:30:00+08) contains 29,860 rows.\n"
    "- Joined analysis found many anomalies: top KPIs show anomaly_count = 30 (i.e., all points in the 30-minute window flagged).\n"
    "  Examples include numerous frontend-0 container metrics (e.g., container_cpu throttling and many fs/io metrics) where p95 and p5 are 0 and all 30 window samples equal that value.\n"
    "- Earliest anomaly timestamps reported = 2022-03-21 07:00:00+08:00 for those flagged KPIs.\n\n"
    "Interpretation / next steps:\n"
    "1) A large number of KPIs are flagged because global p95/p5 thresholds for many metrics are extreme (often 0), causing every window sample to meet the anomaly rule. Verify KPI definitions and whether zero-values are meaningful or indicate missing/idle data.\n"
    "2) Focus follow-up on KPIs with non-trivial deviations (nonzero p95/p5 or large abs(max-p95) or abs(min-p5)) and on frontend containers/nodes shown in the top results.\n"
    "3) Inspect raw metric semantics (e.g., throttling, CPU, disk/fs) and corroborate with traces/logs to determine whether this indicates real resource issues or artefacts of instrumentation.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  count      p95      p5
0     frontend-0         container.node-5.container_cpu_cfs_periods     30  114.900  56.725
1     frontend-0  container.node-5.container_cpu_cfs_throttled_p...     30    0.000   0.000
2     frontend-0  container.node-5.container_cpu_cfs_throttled_s...     30    0.000   0.000
3     frontend-0    container.node-5.container_cpu_load_average_10s     30    0.000   0.000
4     frontend-0      container.node-5.container_cpu_system_seconds     30    0.065   0.035
...          ...                                                ...    ...      ...     ...
1007      node-6                                  system.swap.total     30    0.000   0.000
1008      node-6                                   system.swap.used     30    0.000   0.000
1009      node-6                               system.swap.used_pct     30    0.000   0.000
1010      node-6                             system.tcp.retrans_pct     30    0.000   0.000
1011      node-6                             system.udp.connect.num     30   12.000  10.000

[1012 rows x 5 columns], (29860, 5),        cmdb_id                                           kpi_name  count            p95             p5  anomaly_count       earliest_anomaly_ts  max_value_in_window  min_value_in_window
0   frontend-0  container.node-5.container_cpu_cfs_throttled_p...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
1   frontend-0  container.node-5.container_cpu_cfs_throttled_s...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
2   frontend-0    container.node-5.container_cpu_load_average_10s     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
3   frontend-0        container.node-5.container_file_descriptors     30      17.000000      17.000000             30 2022-03-21 07:00:00+08:00            17.000000            17.000000
4   frontend-0     container.node-5.container_fs_inodes./dev/vda1     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
5   frontend-0  container.node-5.container_fs_inodes_free./dev...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
6   frontend-0  container.node-5.container_fs_io_current./dev/...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
7   frontend-0  container.node-5.container_fs_io_time_seconds....     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
8   frontend-0  container.node-5.container_fs_io_time_weighted...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
9   frontend-0   container.node-5.container_fs_limit_MB./dev/vda1     30  604630.738281  604630.738281             30 2022-03-21 07:00:00+08:00        604630.738281        604630.738281
10  frontend-0  container.node-5.container_fs_read_seconds./de...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
11  frontend-0       container.node-5.container_fs_reads./dev/vda     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
12  frontend-0      container.node-5.container_fs_reads./dev/vda1     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
13  frontend-0    container.node-5.container_fs_reads_MB./dev/vda     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
14  frontend-0  container.node-5.container_fs_reads_merged./de...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
15  frontend-0  container.node-5.container_fs_sector_reads./de...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
16  frontend-0  container.node-5.container_fs_sector_writes./d...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
17  frontend-0  container.node-5.container_fs_write_seconds./d...     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
18  frontend-0      container.node-5.container_fs_writes./dev/vda     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000
19  frontend-0     container.node-5.container_fs_writes./dev/vda1     30       0.000000       0.000000             30 2022-03-21 07:00:00+08:00             0.000000             0.000000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables if present
try:
    df_trace, tz
except NameError:
    df_trace = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) Compute global thresholds (count, p95, p5) for every (cmdb_id, trace_name)
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id','trace_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda s: float(s.quantile(0.95)),
         p5 =lambda s: float(s.quantile(0.05)))
)

# 2) Filter trace.csv to the incident window (inclusive)
start = pd.to_datetime('2022-03-21 07:00:00').tz_localize(tz)
end   = pd.to_datetime('2022-03-21 07:30:00').tz_localize(tz)

trace_window = df_trace[(df_trace['ts'] >= start) & (df_trace['ts'] <= end)].copy()

# 3) Keep only trace_names that contain any of the substrings and join thresholds to window
substrs = ['duration','error_rate','row_count']
mask = trace_thresholds['trace_name'].str.contains('|'.join(substrs), case=False, na=False)
trace_thresholds_filtered = trace_thresholds[mask].copy()

# Filter window to only these trace_names to reduce join size
trace_window_filtered = trace_window[trace_window['trace_name'].str.contains('|'.join(substrs), case=False, na=False)].copy()

# Join thresholds to window rows on (cmdb_id, trace_name)
tw_joined = trace_window_filtered.merge(trace_thresholds_filtered, on=['cmdb_id','trace_name'], how='left', suffixes=('','_global'))

# 4) Per-row anomaly flag using global p95/p5
tw_joined['is_anomaly'] = tw_joined.apply(
    lambda r: (pd.notnull(r['p95']) and r['value'] >= r['p95']) or (pd.notnull(r['p5']) and r['value'] <= r['p5']),
    axis=1
)

# Aggregate per (cmdb_id, trace_name) for window stats
trace_grouped = tw_joined.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# earliest anomaly timestamp per group (if any)
earliest_anom = (
    tw_joined[tw_joined['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts':'earliest_anomaly_ts'})
)

trace_result = trace_grouped.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Ensure timezone for earliest_anomaly_ts
trace_result['earliest_anomaly_ts'] = pd.to_datetime(trace_result['earliest_anomaly_ts']).dt.tz_convert(tz)

# 5) Return top 20 rows sorted by anomaly_count desc, or by deviation if no anomalies
total_anomalies = int(trace_result['anomaly_count'].sum()) if not trace_result.empty else 0

if total_anomalies > 0:
    trace_top20 = trace_result.sort_values(['anomaly_count','count'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    # deviation score
    trace_result['dev_score'] = trace_result.apply(lambda r: max(
        abs(r['max_value_in_window'] - r['p95']) if pd.notnull(r['p95']) else 0,
        abs(r['min_value_in_window'] - r['p5'])  if pd.notnull(r['p5'])  else 0
    ), axis=1)
    trace_top20 = trace_result.sort_values('dev_score', ascending=False).head(20).reset_index(drop=True)
    trace_top20 = trace_top20.drop(columns=['dev_score'], errors='ignore')

# Compact display: selected columns
trace_top20_display = trace_top20[['cmdb_id','trace_name','count','p95','p5','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window']].head(20)

# Expose main intermediate tables and the compact result
trace_thresholds, trace_window.shape, trace_top20_display

```
Out[3]:
```
Summary of trace-level anomaly detection (plain English):

- Data processed: trace_thresholds contains 1,232 (cmdb_id, trace_name) combinations computed from the full series. The incident window filter (2022-03-21 07:00:00+08 to 07:30:00+08) produced trace_window with 25,008 rows.

- Anomalies found: many trace KPIs were flagged as anomalous for the entire 30-minute window (anomaly_count = 30). Examples from the top results include multiple services' error_rate, duration_p95/duration_mean and row_count metrics (e.g., adservice-0 trace.from_frontend-0.error_rate, cartservice-0 trace.from_frontend-1.duration_p95, etc.). For many of these, p95 and p5 are 0 and all window samples are equal to 0 (earliest anomaly timestamp = 2022-03-21 07:00:00+08).

- Likely cause of widespread flags: global thresholds (p95/p5) are frequently zero for many trace metrics, so every zero-valued sample in the window meets the <= p5 or >= p95 rule. This creates many full-window "anomalies" that may reflect metric semantics (idle/no traffic) or instrumentation/normalization artifacts rather than true incidents.

- Recommendations / next steps:
  1. Validate metric semantics: confirm whether zero-valued durations/error_rate/row_count are meaningful or indicate missing/aggregated idle data.
  2. Prioritize KPIs with non-zero p95/p5 or large absolute deviations (max-p95 or min-p5) to surface real deviations in latency or traffic (these are more likely real issues).
  3. Correlate flagged trace KPIs with logs and metrics (CPU, throttling, I/O) for the same services/nodes to determine if anomalies reflect service problems or telemetry artifacts.

If you want, I can (a) list the top deviating trace KPIs by absolute deviation, or (b) show the non-zero p95/p5 anomalies for immediate investigation. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                               trace_name  count        p95        p5
0            adservice-0      trace.from_frontend-0.duration_mean     30   0.000017  0.000013
1            adservice-0       trace.from_frontend-0.duration_p95     30   0.000025  0.000014
2            adservice-0         trace.from_frontend-0.error_rate     30   0.000000  0.000000
3            adservice-0          trace.from_frontend-0.row_count     30   7.000000  2.000000
4            adservice-0      trace.from_frontend-1.duration_mean     29   0.000017  0.000012
...                  ...                                      ...    ...        ...       ...
1227  shippingservice2-0  trace.from_checkoutservice2-0.row_count     28   8.000000  4.000000
1228  shippingservice2-0     trace.from_frontend2-0.duration_mean     30   0.000085  0.000059
1229  shippingservice2-0      trace.from_frontend2-0.duration_p95     30   0.000184  0.000064
1230  shippingservice2-0        trace.from_frontend2-0.error_rate     30   0.000000  0.000000
1231  shippingservice2-0         trace.from_frontend2-0.row_count     30  20.100000  7.450000

[1232 rows x 5 columns], (25008, 5),           cmdb_id                           trace_name  count       p95   p5  anomaly_count       earliest_anomaly_ts  max_value_in_window  min_value_in_window
0     adservice-0     trace.from_frontend-0.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
1     adservice-0     trace.from_frontend-2.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
2     adservice-1     trace.from_frontend-0.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
3     adservice-2     trace.from_frontend-0.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
4     adservice-2     trace.from_frontend-1.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
5    adservice2-0    trace.from_frontend2-0.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
6   cartservice-0     trace.from_frontend-0.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
7   cartservice-0  trace.from_frontend-1.duration_mean     30  0.000110  0.0             30 2022-03-21 07:00:00+08:00             0.000500                  0.0
8   cartservice-0   trace.from_frontend-1.duration_p95     30  0.000440  0.0             30 2022-03-21 07:00:00+08:00             0.000950                  0.0
9   cartservice-0     trace.from_frontend-1.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
10  cartservice-0  trace.from_frontend-2.duration_mean     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000250                  0.0
11  cartservice-0   trace.from_frontend-2.duration_p95     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000850                  0.0
12  cartservice-0     trace.from_frontend-2.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
13  cartservice-0             trace.self.duration_mean     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
14  cartservice-0              trace.self.duration_p95     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
15  cartservice-0                trace.self.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
16  cartservice-1  trace.from_frontend-0.duration_mean     30  0.000092  0.0             30 2022-03-21 07:00:00+08:00             0.000333                  0.0
17  cartservice-1   trace.from_frontend-0.duration_p95     30  0.000412  0.0             30 2022-03-21 07:00:00+08:00             0.000900                  0.0
18  cartservice-1     trace.from_frontend-0.error_rate     30  0.000000  0.0             30 2022-03-21 07:00:00+08:00             0.000000                  0.0
19  cartservice-1  trace.from_frontend-1.duration_mean     30  0.000110  0.0             30 2022-03-21 07:00:00+08:00             0.000250                  0.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables if available, otherwise load
try:
    df_err, df_log, tz
except NameError:
    df_err = pd.read_csv('error_logs.csv')
    df_log = pd.read_csv('log.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts columns exist and are timezone-aware (Asia/Shanghai)
if 'ts' not in df_err.columns:
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
if 'ts' not in df_log.columns:
    df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Targets (frontend pods)
frontends = ['frontend-0','frontend-1','frontend-2']

# Incident window (inclusive) in UTC+8
start = pd.to_datetime('2022-03-21 07:00:00').tz_localize(tz)
end   = pd.to_datetime('2022-03-21 07:30:00').tz_localize(tz)

# 1) error_logs analysis
# Full-file per-cmdb_id counts and earliest/latest timestamps
error_full_summary = (
    df_err
    .groupby('cmdb_id', as_index=False)
    .agg(total_errors_full = ('message','count'),
         earliest_full = ('ts','min'),
         latest_full = ('ts','max'))
)

# Filter error_logs to incident window
err_window = df_err[(df_err['ts'] >= start) & (df_err['ts'] <= end)].copy()

# For each frontend pod: total error rows in-window, earliest in-window ts, latest in-window ts
error_frontend_window_summary = (
    err_window[err_window['cmdb_id'].isin(frontends)]
    .groupby('cmdb_id', as_index=False)
    .agg(total_errors_in_window = ('message','count'),
         earliest_in_window = ('ts','min'),
         latest_in_window = ('ts','max'))
)

# Ensure all frontends appear (even if zero)
missing = [f for f in frontends if f not in error_frontend_window_summary['cmdb_id'].values]
if missing:
    df_missing = pd.DataFrame({
        'cmdb_id': missing,
        'total_errors_in_window': 0,
        'earliest_in_window': pd.NaT,
        'latest_in_window': pd.NaT
    })
    error_frontend_window_summary = pd.concat([error_frontend_window_summary, df_missing], ignore_index=True)

# Up to 20 sample message strings for frontend pods within the incident window
error_messages_sample = (
    err_window[err_window['cmdb_id'].isin(frontends)]
    .loc[:, ['cmdb_id','message','ts']]
    .dropna(subset=['message'])
    .sort_values(['cmdb_id','ts'])
    .head(200)  # take more then dedupe to ensure variety
)
# Keep unique messages up to 20
error_messages_sample = (error_messages_sample['message'].drop_duplicates().head(20).reset_index(drop=True).to_frame(name='sample_message'))

# 2) log.csv analysis for frontends
# Compute global thresholds (full series) for each (cmdb_id, log_name)
log_thresholds = (
    df_log
    .groupby(['cmdb_id','log_name'], as_index=False)['value']
    .agg(count='count',
         p95=lambda s: float(s.quantile(0.95)),
         p5 =lambda s: float(s.quantile(0.05)))
)

# Filter log.csv to frontend pods and incident window
log_window = df_log[(df_log['cmdb_id'].isin(frontends)) & (df_log['ts'] >= start) & (df_log['ts'] <= end)].copy()

# Join thresholds to window rows on (cmdb_id, log_name)
lw_joined = log_window.merge(log_thresholds, on=['cmdb_id','log_name'], how='left', suffixes=('','_global'))

# Per-row anomaly flag using global p95/p5 (respect global thresholds even if NaN)
lw_joined['is_anomaly'] = (
    (lw_joined['p95'].notnull() & (lw_joined['value'] >= lw_joined['p95'])) |
    (lw_joined['p5'].notnull()  & (lw_joined['value'] <= lw_joined['p5']))
)

# Aggregate per (cmdb_id, log_name)
log_grouped = lw_joined.groupby(['cmdb_id','log_name'], as_index=False).agg(
    count = ('value','count'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# earliest anomaly timestamp per group (if any)
earliest_anom_log = (
    lw_joined[lw_joined['is_anomaly']]
    .groupby(['cmdb_id','log_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts':'earliest_anomaly_ts'})
)

log_result = log_grouped.merge(earliest_anom_log, on=['cmdb_id','log_name'], how='left')

# Ensure timezone for earliest_anomaly_ts
log_result['earliest_anomaly_ts'] = pd.to_datetime(log_result['earliest_anomaly_ts']).dt.tz_convert(tz)

# Determine if any anomalies exist; if none, compute deviation score for fallback ordering
if log_result['anomaly_count'].sum() > 0:
    log_result_top = log_result.sort_values(['anomaly_count','count'], ascending=[False, False]).head(20).reset_index(drop=True)
else:
    log_result['dev_score'] = log_result.apply(lambda r: max(
        abs(r['max_value_in_window'] - r['p95']) if pd.notnull(r['p95']) else 0,
        abs(r['min_value_in_window'] - r['p5'])  if pd.notnull(r['p5'])  else 0
    ), axis=1)
    log_result_top = log_result.sort_values('dev_score', ascending=False).head(20).reset_index(drop=True)
    log_result_top = log_result_top.drop(columns=['dev_score'], errors='ignore')

# Compact display: selected columns for top results
log_result_display = log_result_top[['cmdb_id','log_name','count','p95','p5','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window']].head(20)

# Expose intermediate tables and compact outputs for controller
error_full_summary, error_frontend_window_summary, error_messages_sample, log_thresholds, log_window.shape, log_result_display

```
Out[4]:
```
```python
# Summarize findings for frontend pods using kernel-stored results
summary = (
    "Frontend-focused findings (2022-03-21 07:00:00+08 to 07:30:00+08):\n\n"
    "Error logs (full-file & in-window):\n"
    f" - frontend-0: 158 error rows (in-window). Earliest: 2022-03-21 07:00:09+08, Latest: 2022-03-21 07:29:55+08.\n"
    f" - frontend-1: 82 error rows (in-window). Earliest: 2022-03-21 07:00:00+08, Latest: 2022-03-21 07:29:30+08.\n"
    f" - frontend-2: 79 error rows (in-window). Earliest: 2022-03-21 07:00:23+08, Latest: 2022-03-21 07:29:16+08.\n\n"
    "Sample error message(s) from frontends during the window (up to 20):\n"
    f" - {error_messages_sample['sample_message'].iloc[0] if not error_messages_sample.empty else 'No sample messages found.'}\n\n"
    "Log metrics (global thresholds used; window analysis & anomalies):\n"
    " - log.error_count anomalies (anomaly = value >= global P95 OR <= global P5):\n"
    f"   * frontend-0: count=30 points, anomaly_count=13, earliest_anomaly=2022-03-21 07:00:00+08, max=8, min=2\n"
    f"   * frontend-2: count=30 points, anomaly_count=9,  earliest_anomaly=2022-03-21 07:00:00+08, max=6, min=1\n"
    f"   * frontend-1: count=30 points, anomaly_count=7,  earliest_anomaly=2022-03-21 07:06:00+08, max=6, min=1\n\n"
    " - log.row_count anomalies (same rule):\n"
    f"   * frontend-0: count=30, anomaly_count=4, earliest_anomaly=2022-03-21 07:08:00+08, max=508, min=202\n"
    f"   * frontend-1: count=30, anomaly_count=4, earliest_anomaly=2022-03-21 07:04:00+08, max=282, min=106\n"
    f"   * frontend-2: count=30, anomaly_count=4, earliest_anomaly=2022-03-21 07:10:00+08, max=257, min=108\n\n"
    "Interpretation and recommended next steps:\n"
    "1) Frontend pods show substantial error log volume during the incident window (hundreds of error rows each). Prioritize investigating frontend-0 first (largest count).\n"
    "2) log.error_count shows repeated anomalies starting at the start of the window for multiple frontends — inspect the sample error messages and application logs for root causes (exceptions, upstream failures, or request handling errors).\n"
    "3) Correlate these frontend log anomalies with traces and the metric anomalies previously identified (CPU, throttling, fs/io) to determine whether errors are resource-related or application-level.\n"
    "4) If you want, I can (a) dump the up-to-20 sample error messages, (b) list full per-frontend log time series in-window, or (c) correlate errors with trace spans for top offending timestamps.\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_errors_full             earliest_full               latest_full
0  frontend-0                158 2022-03-21 07:00:09+08:00 2022-03-21 07:29:55+08:00
1  frontend-1                 82 2022-03-21 07:00:00+08:00 2022-03-21 07:29:30+08:00
2  frontend-2                 79 2022-03-21 07:00:23+08:00 2022-03-21 07:29:16+08:00,       cmdb_id  total_errors_in_window        earliest_in_window          latest_in_window
0  frontend-0                     158 2022-03-21 07:00:09+08:00 2022-03-21 07:29:55+08:00
1  frontend-1                      82 2022-03-21 07:00:00+08:00 2022-03-21 07:29:30+08:00
2  frontend-2                      79 2022-03-21 07:00:23+08:00 2022-03-21 07:29:16+08:00,                                       sample_message
0  severity: warning, message: failed to retrieve...,               cmdb_id         log_name  count    p95     p5
0         adservice-0  log.error_count     30    0.0    0.0
1         adservice-0    log.row_count     30  318.0  122.9
2         adservice-1  log.error_count     30    0.0    0.0
3         adservice-1    log.row_count     30   26.0   10.0
4         adservice-2  log.error_count     30    0.0    0.0
..                ...              ...    ...    ...    ...
57  shippingservice-0    log.row_count     30   33.0    4.0
58  shippingservice-1  log.error_count     29    0.0    0.0
59  shippingservice-1    log.row_count     29   36.0    4.0
60  shippingservice-2  log.error_count     29    0.0    0.0
61  shippingservice-2    log.row_count     29   32.6    4.0

[62 rows x 5 columns], (180, 5),       cmdb_id         log_name  count     p95      p5  anomaly_count       earliest_anomaly_ts  max_value_in_window  min_value_in_window
0  frontend-0  log.error_count     30    7.00    3.00             13 2022-03-21 07:00:00+08:00                  8.0                  2.0
1  frontend-2  log.error_count     30    4.55    1.00              9 2022-03-21 07:00:00+08:00                  6.0                  1.0
2  frontend-1  log.error_count     30    4.55    1.00              7 2022-03-21 07:06:00+08:00                  6.0                  1.0
3  frontend-0    log.row_count     30  491.40  212.75              4 2022-03-21 07:08:00+08:00                508.0                202.0
4  frontend-1    log.row_count     30  261.65  115.90              4 2022-03-21 07:04:00+08:00                282.0                106.0
5  frontend-2    log.row_count     30  244.55  115.90              4 2022-03-21 07:10:00+08:00                257.0                108.0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables where available
try:
    trace_thresholds, trace_window, tz
except NameError:
    # load if not present (fallback, but should be present)
    trace_thresholds = pd.read_csv('trace.csv')  # placeholder to avoid NameError
    trace_window = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')

# Define substrings to filter trace names
substrs = ['from_frontend','from_frontend2','to_','self','error_rate','duration','row_count']
pattern = '|'.join(substrs)

# Filter trace_thresholds for relevant trace_names (global thresholds)
trace_thresholds_filtered = trace_thresholds[trace_thresholds['trace_name'].str.contains(pattern, case=False, na=False)].copy()

# Filter trace_window (incident-window rows) for relevant trace_names
trace_window_filtered = trace_window[trace_window['trace_name'].str.contains(pattern, case=False, na=False)].copy()

# Join thresholds to window rows on (cmdb_id, trace_name)
tw_join = trace_window_filtered.merge(
    trace_thresholds_filtered,
    on=['cmdb_id','trace_name'],
    how='left',
    suffixes=('','_global')
)

# Per-row anomaly flag using global p95/p5 (only flag if thresholds exist)
tw_join['is_anomaly'] = (
    tw_join['p95'].notnull() & (tw_join['value'] >= tw_join['p95'])
) | (
    tw_join['p5'].notnull() & (tw_join['value'] <= tw_join['p5'])
)

# Aggregate required metrics per (cmdb_id, trace_name)
agg = tw_join.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count = ('value','count'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    anomaly_count = ('is_anomaly','sum'),
    earliest_anomaly_ts = ('ts', lambda s: s[tw_join.loc[s.index,'is_anomaly']].min() if any(tw_join.loc[s.index,'is_anomaly']) else pd.NaT),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# Ensure earliest_anomaly_ts is timezone-aware in Asia/Shanghai (if not NaT)
agg['earliest_anomaly_ts'] = pd.to_datetime(agg['earliest_anomaly_ts']).dt.tz_convert(tz)

# Compute deviation = max(abs(max - p95), abs(min - p5)) with safe handling of missing p95/p5
def compute_dev(row):
    dev_candidates = []
    if pd.notnull(row['p95']):
        dev_candidates.append(abs(row['max_value_in_window'] - row['p95']))
    if pd.notnull(row['p5']):
        dev_candidates.append(abs(row['min_value_in_window'] - row['p5']))
    return float(max(dev_candidates)) if dev_candidates else 0.0

agg['deviation'] = agg.apply(compute_dev, axis=1)

# Sort by deviation desc, tie-breaker anomaly_count desc
trace_deviation_top20 = agg.sort_values(['deviation','anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Select and order requested columns compactly
trace_deviation_top20 = trace_deviation_top20[[
    'cmdb_id','trace_name','count','p95','p5',
    'max_value_in_window','min_value_in_window','deviation','anomaly_count','earliest_anomaly_ts'
]]

# Display compact top-20 result
trace_deviation_top20

```
Out[5]:
```
Summary of trace deviation ranking (top 20):

- The largest deviations are in row_count metrics involving frontend2 <-> currencyservice2:
  - currencyservice2-0 / trace.from_frontend2-0.row_count — count=30, p95=248.85, p5=97.50, max=293, min=71, deviation=44.15, anomaly_count=4, earliest anomaly=2022-03-21 07:07:00+08.
  - frontend2-0 / trace.to_currencyservice2-0.row_count — same stats (mirror of above), deviation=44.15, earliest anomaly=07:07.

- Other notable high deviations:
  - frontend2-0 / trace.self.row_count — count=30, p95=843.00, p5=324.20, max=870, min=306, deviation=27.00, anomaly_count=4, earliest anomaly=07:10.
  - checkoutservice2-0 / trace.self.row_count — deviation=15.00, anomaly_count=7, earliest anomaly=07:02.
  - frontend2-0 / trace.to_productcatalogservice2-0.row_count and productcatalogservice2-0 / trace.from_frontend2-0.row_count — deviation≈14.10, anomaly_count=4, earliest anomaly=07:08.
  - Several frontend and cartservice self.row_count entries show deviations 6–12 with anomaly counts 4–5 (earliest anomalies mostly between ~07:01–07:07).

Interpretation and next steps:

1. Highest deviations center on frontend2 and currencyservice2 interactions (row_count increases above historical p95). These are the top candidates for investigation.
2. Some services show elevated self.row_count (frontend2, frontend-0, frontend-1, frontend-2, cartservice2-0) indicating increased request/throughput levels during the window; checkoutservice2-0 shows early anomalies (~07:02).
3. Anomaly counts are small relative to window length (e.g., 4–7 of 30 samples), so issues appear as short bursts rather than sustained across the whole window.
4. Recommended actions: correlate these trace anomalies with frontend logs and metric anomalies (CPU, throttling, I/O) at the reported earliest timestamps (notably ~07:02, ~07:07, ~07:08, ~07:10) to determine cause (traffic surge, upstream failures, or instrumentation effects).
5. Prioritize investigating frontend2 ↔ currencyservice2 and frontend2 overall, then checkoutservice2-0 and productcatalogservice2-0 based on deviation magnitude and anomaly timing.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                   trace_name  count     p95      p5  max_value_in_window  min_value_in_window  deviation  anomaly_count       earliest_anomaly_ts
0         currencyservice2-0             trace.from_frontend2-0.row_count     30  248.85   97.50                293.0                 71.0      44.15              4 2022-03-21 07:07:00+08:00
1                frontend2-0        trace.to_currencyservice2-0.row_count     30  248.85   97.50                293.0                 71.0      44.15              4 2022-03-21 07:07:00+08:00
2                frontend2-0                         trace.self.row_count     30  843.00  324.20                870.0                306.0      27.00              4 2022-03-21 07:10:00+08:00
3         checkoutservice2-0                         trace.self.row_count     28   32.00   15.35                 47.0                  9.0      15.00              7 2022-03-21 07:02:00+08:00
4                frontend2-0  trace.to_productcatalogservice2-0.row_count     30  408.95  155.10                420.0                141.0      14.10              4 2022-03-21 07:08:00+08:00
5   productcatalogservice2-0             trace.from_frontend2-0.row_count     30  408.95  155.10                420.0                141.0      14.10              4 2022-03-21 07:08:00+08:00
6                 frontend-1                         trace.self.row_count     30  169.05   57.75                181.0                 50.0      11.95              4 2022-03-21 07:04:00+08:00
7             cartservice2-0                         trace.self.row_count     30   88.00   34.90                 99.0                 32.0      11.00              5 2022-03-21 07:04:00+08:00
8                 frontend-0                         trace.self.row_count     30  311.65  114.25                316.0                108.0       6.25              4 2022-03-21 07:06:00+08:00
9         checkoutservice2-0                    trace.from_root.row_count     28   12.00    5.35                 18.0                  3.0       6.00              7 2022-03-21 07:02:00+08:00
10        checkoutservice2-0        trace.to_currencyservice2-0.row_count     28   12.00    5.35                 18.0                  3.0       6.00              7 2022-03-21 07:02:00+08:00
11        currencyservice2-0      trace.from_checkoutservice2-0.row_count     28   12.00    5.35                 18.0                  3.0       6.00              7 2022-03-21 07:02:00+08:00
12              adservice2-0             trace.from_frontend2-0.row_count     30   51.00   21.45                 52.0                 16.0       5.45              6 2022-03-21 07:00:00+08:00
13               frontend2-0              trace.to_adservice2-0.row_count     30   51.00   21.45                 52.0                 16.0       5.45              6 2022-03-21 07:00:00+08:00
14                frontend-2                         trace.self.row_count     30  159.00   58.45                162.0                 54.0       4.45              5 2022-03-21 07:01:00+08:00
15         currencyservice-0              trace.from_frontend-0.row_count     30   32.55   12.35                 36.0                  8.0       4.35              4 2022-03-21 07:06:00+08:00
16         currencyservice-1              trace.from_frontend-0.row_count     30   33.10   12.35                 35.0                  8.0       4.35              4 2022-03-21 07:06:00+08:00
17                frontend-0         trace.to_currencyservice-0.row_count     30   32.55   12.35                 36.0                  8.0       4.35              4 2022-03-21 07:06:00+08:00
18                frontend-0         trace.to_currencyservice-1.row_count     30   33.10   12.35                 35.0                  8.0       4.35              4 2022-03-21 07:06:00+08:00
19                frontend-0   trace.to_productcatalogservice-1.row_count     30   46.55   17.25                 48.0                 13.0       4.25              4 2022-03-21 07:08:00+08:00```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse kernel-stored df_err and tz if present; otherwise load and prepare
try:
    df_err, tz
except NameError:
    df_err = pd.read_csv('error_logs.csv')
    tz = pytz.timezone('Asia/Shanghai')
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Ensure ts exists and is timezone-aware Asia/Shanghai
if 'ts' not in df_err.columns:
    tz = pytz.timezone('Asia/Shanghai')
    df_err['ts'] = pd.to_datetime(df_err['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Targets and incident window (inclusive)
frontends = ['frontend-0','frontend-1','frontend-2']
start = pd.to_datetime('2022-03-21 07:00:00').tz_localize(tz)
end   = pd.to_datetime('2022-03-21 07:30:00').tz_localize(tz)

# Filter to frontends within the incident window
err_window_front = df_err[
    df_err['cmdb_id'].isin(frontends) &
    (df_err['ts'] >= start) &
    (df_err['ts'] <= end)
].copy()

# 1) Per-frontend summary table
error_frontend_summary = (
    err_window_front
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_error_rows_in_window = ('message','count'),
        earliest_in_window_timestamp = ('ts','min'),
        latest_in_window_timestamp = ('ts','max')
    )
)

# Ensure all frontends present (include zeros if missing)
for f in frontends:
    if f not in error_frontend_summary['cmdb_id'].values:
        error_frontend_summary = pd.concat([
            error_frontend_summary,
            pd.DataFrame([{
                'cmdb_id': f,
                'total_error_rows_in_window': 0,
                'earliest_in_window_timestamp': pd.NaT,
                'latest_in_window_timestamp': pd.NaT
            }])
        ], ignore_index=True)

# Order rows by frontend name for consistency
error_frontend_summary = error_frontend_summary.set_index('cmdb_id').loc[frontends].reset_index()

# 2) Per-minute aggregated error counts across the window for the frontends
# Create full minute index inclusive of start and end
minute_index = pd.date_range(start=start, end=end, freq='T', tz=tz)

# Floor each error ts to minute
err_window_front['minute'] = err_window_front['ts'].dt.floor('T')

# Group counts per minute and cmdb_id, then pivot
per_minute_counts = (
    err_window_front
    .groupby(['minute','cmdb_id'], as_index=False)
    .size()
    .rename(columns={'size':'count'})
    .pivot(index='minute', columns='cmdb_id', values='count')
    .reindex(minute_index)  # ensure full minute range
    .fillna(0)
    .astype(int)
)

# Ensure columns for each frontend in specified order
for f in frontends:
    if f not in per_minute_counts.columns:
        per_minute_counts[f] = 0
per_minute_counts = per_minute_counts[frontends].reset_index().rename(columns={'index':'minute_timestamp'})

# 3) Up to 20 sample error log rows sorted by timestamp ascending (timestamp, cmdb_id, message)
error_samples = (
    err_window_front
    .loc[:, ['ts','cmdb_id','message']]
    .sort_values('ts', ascending=True)
    .head(20)
    .rename(columns={'ts':'timestamp'})
    .reset_index(drop=True)
)

# Compact outputs (store full intermediates in variables above)
error_frontend_summary, per_minute_counts, error_samples

```
Out[6]:
```
Summary (plain English):

- Per-frontend totals (incident window 2022-03-21 07:00:00+08 to 07:30:00+08):
  - frontend-0: 158 error rows. Earliest: 2022-03-21 07:00:09+08, Latest: 2022-03-21 07:29:55+08.
  - frontend-1: 82 error rows. Earliest: 2022-03-21 07:00:00+08, Latest: 2022-03-21 07:29:30+08.
  - frontend-2: 79 error rows. Earliest: 2022-03-21 07:00:23+08, Latest: 2022-03-21 07:29:16+08.

- Per-minute aggregated error counts (30-minute window, minute-resolution):
  - Counts are present for every minute from 07:00 through 07:29 (07:30 shows zeros).
  - Error volume fluctuates minute-to-minute. Observed per-minute peaks in the window:
    - frontend-0: up to 8 errors in a single minute (e.g., 07:29).
    - frontend-1: up to 6 errors in a single minute.
    - frontend-2: up to 6 errors in a single minute.
  - Errors are sustained across the window (not a single spike) with per-minute counts in most minutes.

- Sample error messages (up to 20, earliest-first):
  - The sampled entries are dominated by a repeated warning text: "severity: warning, message: failed to retrieve..." (many identical/near-identical messages across frontends and times).

Actionable interpretation and next steps:
1. Frontend-0 is the highest volume offender and should be prioritized for investigation.
2. The recurring "failed to retrieve..." warnings across frontends indicate a common failure mode (likely upstream/timeouts or resource contention). Inspect full message details and stack traces for that warning to identify the failing call or dependency.
3. Correlate the high-minute error bursts (notably around 07:00–07:02, ~07:07, ~07:27–07:29) with trace and metric anomalies (latency, row_count, CPU/throttling, I/O) to determine if errors are caused by traffic surge, upstream service failures, or resource exhaustion.
4. If you want, I can (a) show the full 30-minute per-minute table, (b) expand the 20 sample messages to include full message text, or (c) correlate specific error timestamps with trace spans and metric anomalies. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id  total_error_rows_in_window earliest_in_window_timestamp latest_in_window_timestamp
0  frontend-0                         158    2022-03-21 07:00:09+08:00  2022-03-21 07:29:55+08:00
1  frontend-1                          82    2022-03-21 07:00:00+08:00  2022-03-21 07:29:30+08:00
2  frontend-2                          79    2022-03-21 07:00:23+08:00  2022-03-21 07:29:16+08:00, cmdb_id          minute_timestamp  frontend-0  frontend-1  frontend-2
0       2022-03-21 07:00:00+08:00           3           3           1
1       2022-03-21 07:01:00+08:00           7           3           5
2       2022-03-21 07:02:00+08:00           2           2           1
3       2022-03-21 07:03:00+08:00           7           3           3
4       2022-03-21 07:04:00+08:00           4           3           1
..                            ...         ...         ...         ...
26      2022-03-21 07:26:00+08:00           5           2           2
27      2022-03-21 07:27:00+08:00           7           5           6
28      2022-03-21 07:28:00+08:00           6           6           4
29      2022-03-21 07:29:00+08:00           8           1           1
30      2022-03-21 07:30:00+08:00           0           0           0

[31 rows x 4 columns],                    timestamp     cmdb_id                                            message
0  2022-03-21 07:00:00+08:00  frontend-1  severity: warning, message: failed to retrieve...
1  2022-03-21 07:00:09+08:00  frontend-0  severity: warning, message: failed to retrieve...
2  2022-03-21 07:00:14+08:00  frontend-1  severity: warning, message: failed to retrieve...
3  2022-03-21 07:00:19+08:00  frontend-1  severity: warning, message: failed to retrieve...
4  2022-03-21 07:00:23+08:00  frontend-2  severity: warning, message: failed to retrieve...
5  2022-03-21 07:00:28+08:00  frontend-0  severity: warning, message: failed to retrieve...
6  2022-03-21 07:00:59+08:00  frontend-0  severity: warning, message: failed to retrieve...
7  2022-03-21 07:01:00+08:00  frontend-2  severity: warning, message: failed to retrieve...
8  2022-03-21 07:01:08+08:00  frontend-1  severity: warning, message: failed to retrieve...
9  2022-03-21 07:01:10+08:00  frontend-0  severity: warning, message: failed to retrieve...
10 2022-03-21 07:01:15+08:00  frontend-2  severity: warning, message: failed to retrieve...
11 2022-03-21 07:01:16+08:00  frontend-0  severity: warning, message: failed to retrieve...
12 2022-03-21 07:01:24+08:00  frontend-0  severity: warning, message: failed to retrieve...
13 2022-03-21 07:01:25+08:00  frontend-2  severity: warning, message: failed to retrieve...
14 2022-03-21 07:01:33+08:00  frontend-1  severity: warning, message: failed to retrieve...
15 2022-03-21 07:01:34+08:00  frontend-0  severity: warning, message: failed to retrieve...
16 2022-03-21 07:01:39+08:00  frontend-2  severity: warning, message: failed to retrieve...
17 2022-03-21 07:01:42+08:00  frontend-0  severity: warning, message: failed to retrieve...
18 2022-03-21 07:01:50+08:00  frontend-0  severity: warning, message: failed to retrieve...
19 2022-03-21 07:01:53+08:00  frontend-1  severity: warning, message: failed to retrieve...)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables
try:
    trace_thresholds, trace_window, tz
except NameError:
    # fallback load (should not be needed in this environment)
    trace_thresholds = pd.read_csv('trace.csv')
    trace_window = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')
    trace_window['ts'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate components and name filter
candidates = ['currencyservice2-0','productcatalogservice2-0','checkoutservice2-0','frontend2-0']
pattern = 'duration|error_rate'

# 1) Filter global thresholds for the candidate components and desired trace_name patterns
thresholds_filtered = trace_thresholds[
    trace_thresholds['cmdb_id'].isin(candidates) &
    trace_thresholds['trace_name'].str.contains(pattern, case=False, na=False)
].copy()

# 2) Filter window rows for the same components and patterns
window_filtered = trace_window[
    trace_window['cmdb_id'].isin(candidates) &
    trace_window['trace_name'].str.contains(pattern, case=False, na=False)
].copy()

# 3) Join thresholds to window rows (do NOT recompute thresholds)
joined = window_filtered.merge(
    thresholds_filtered,
    on=['cmdb_id','trace_name'],
    how='left',
    suffixes=('','_global')
)

# 4) Per-row anomaly flag using global p95/p5
joined['is_anomaly'] = (
    joined['p95'].notnull() & (joined['value'] >= joined['p95'])
) | (
    joined['p5'].notnull() & (joined['value'] <= joined['p5'])
)

# 5) Aggregate required metrics per (cmdb_id, trace_name)
agg_basic = joined.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count = ('value','count'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min'),
    anomaly_count = ('is_anomaly','sum'),
    p95 = ('p95','first'),
    p5  = ('p5','first')
)

# earliest anomaly timestamp per group (if any)
anoms = joined[joined['is_anomaly']].groupby(['cmdb_id','trace_name'], as_index=False)['ts'].min().rename(columns={'ts':'earliest_anomaly_timestamp_in_window'})

agg = agg_basic.merge(anoms, on=['cmdb_id','trace_name'], how='left')

# Ensure timezone-aware earliest_anomaly column
agg['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(agg['earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)

# 6) Compute deviation for fallback ranking if needed
def compute_deviation(r):
    devs = []
    if pd.notnull(r['p95']):
        devs.append(abs(r['max_value_in_window'] - r['p95']))
    if pd.notnull(r['p5']):
        devs.append(abs(r['min_value_in_window'] - r['p5']))
    return float(max(devs)) if devs else 0.0

agg['deviation'] = agg.apply(compute_deviation, axis=1)

# 7) Decide sorting: if any anomalies exist, sort by earliest_anomaly_timestamp ascending (NaT last).
# Otherwise return top 20 by deviation desc (tie-breaker anomaly_count desc)
total_anoms = int(agg['anomaly_count'].sum()) if not agg.empty else 0

if total_anoms > 0:
    result = agg.sort_values(['earliest_anomaly_timestamp_in_window','anomaly_count'], ascending=[True, False], na_position='last').head(20).reset_index(drop=True)
else:
    result = agg.sort_values(['deviation','anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)

# Select and order requested columns compactly
result_display = result[[
    'cmdb_id','trace_name','count','p95','p5',
    'anomaly_count','earliest_anomaly_timestamp_in_window',
    'max_value_in_window','min_value_in_window'
]].head(20)

# Expose intermediate variables and compact result for controller
thresholds_filtered, window_filtered.shape, result_display

```
Out[7]:
```
Summary (plain English):

- Scope & data: I analyzed trace KPIs for the candidate components currencyservice2-0, productcatalogservice2-0, checkoutservice2-0, and frontend2-0, restricted to trace_names containing "duration" or "error_rate". The joined window contains 1,908 trace rows and global thresholds were available for 66 (cmdb_id, trace_name) combos.

- Main finding: Many trace KPIs are flagged as anomalies immediately at the start of the window (earliest anomaly timestamps mostly 2022-03-21 07:00:00+08). A large portion of the top results are error_rate series with global p95/p5 = 0, so every zero-valued sample in the window is being counted as an "anomaly" (anomaly_count equals the window sample count for those series). Examples:
  - currencyservice2-0 / trace.from_frontend2-0.error_rate — count=30, p95=0, p5=0, anomaly_count=30 (earliest 07:00).
  - frontend2-0 / many trace.*.error_rate entries — all zero thresholds, anomaly_count=30 (earliest 07:00).
  - checkoutservice2-0 / several trace.*.error_rate entries — anomaly_count equal to count (earliest 07:00).

- Notable non-error_rate deviations (useful signals):
  - frontend2-0 / trace.self.duration_p95 and trace.self.duration_mean show actual non-zero p95/p5 and measurable deviations (e.g., trace.self.row_count and durations seen earlier in previous deviation ranking). Some duration metrics have non-zero p95 and show early anomalies (e.g., trace.to_cartservice2-0.duration_p95 with p95=0.001 and anomaly_count=30 at 07:00).
  - checkoutservice2-0 duration metrics have small but non-zero p95/p5 and show anomalous samples beginning ~07:00 (these may reflect latency shifts).

- Interpretation:
  - A lot of the flagged “anomalies” are artifacts caused by zero-valued global thresholds (p95/p5 = 0) for error_rate series; they create full-window flags but likely indicate lack-of-traffic or metric semantics rather than a true incident.
  - The most actionable signals are duration metrics and row_count metrics with non-zero thresholds and significant deviation. These point to increased latency or changed request counts for frontend2 and its downstream services (currencyservice2-0, productcatalogservice2-0, checkoutservice2-0).

- Recommended next steps:
  1. Prioritize investigation of frontend2-0 <-> currencyservice2-0 and frontend2-0 self-duration/row_count anomalies (they showed high deviation in earlier analysis).
  2. For the error_rate series with zero thresholds, validate metric semantics (are zeros meaningful or just no traffic?) before acting. Treat zero-threshold full-window flags as low-priority unless corroborated by non-zero deviations.
  3. Correlate the earliest anomaly timestamps (notably ~07:00 to 07:10) with frontend error logs and the metric anomalies (CPU, throttling, I/O) to determine root cause (traffic surge, upstream failures, or resource issues).
  4. If you want, I can list only trace duration series with non-zero p95/p5 and non-trivial deviations for focused inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                  trace_name  count       p95        p5
412         checkoutservice2-0        trace.from_frontend2-0.duration_mean     28  0.085707  0.040225
413         checkoutservice2-0         trace.from_frontend2-0.duration_p95     28  0.095899  0.040957
414         checkoutservice2-0           trace.from_frontend2-0.error_rate     28  0.000000  0.000000
416         checkoutservice2-0               trace.from_root.duration_mean     28  0.017412  0.002819
417         checkoutservice2-0                trace.from_root.duration_p95     28  0.052153  0.003476
...                        ...                                         ...    ...       ...       ...
1041  productcatalogservice2-0  trace.from_checkoutservice2-0.duration_p95     28  0.005663  0.004425
1042  productcatalogservice2-0    trace.from_checkoutservice2-0.error_rate     28  0.000000  0.000000
1044  productcatalogservice2-0        trace.from_frontend2-0.duration_mean     30  0.004537  0.003849
1045  productcatalogservice2-0         trace.from_frontend2-0.duration_p95     30  0.005680  0.005092
1046  productcatalogservice2-0           trace.from_frontend2-0.error_rate     30  0.000000  0.000000

[66 rows x 5 columns], (1908, 5),                      cmdb_id                                    trace_name  count       p95     p5  anomaly_count earliest_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window
0         currencyservice2-0             trace.from_frontend2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
1                frontend2-0                    trace.from_root.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
2                frontend2-0              trace.to_adservice2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
3                frontend2-0          trace.to_cartservice2-0.duration_p95     30  0.001000  0.001             30            2022-03-21 07:00:00+08:00              0.00100                0.001
4                frontend2-0            trace.to_cartservice2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
5                frontend2-0        trace.to_currencyservice2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
6                frontend2-0  trace.to_productcatalogservice2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
7                frontend2-0  trace.to_recommendationservice2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
8                frontend2-0        trace.to_shippingservice2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
9   productcatalogservice2-0             trace.from_frontend2-0.error_rate     30  0.000000  0.000             30            2022-03-21 07:00:00+08:00              0.00000                0.000
10        checkoutservice2-0             trace.from_frontend2-0.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
11        checkoutservice2-0                    trace.from_root.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
12        checkoutservice2-0                         trace.self.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
13        checkoutservice2-0         trace.to_cartservice2-0.duration_mean     28  0.000167  0.000             28            2022-03-21 07:00:00+08:00              0.00025                0.000
14        checkoutservice2-0          trace.to_cartservice2-0.duration_p95     28  0.000750  0.000             28            2022-03-21 07:00:00+08:00              0.00085                0.000
15        checkoutservice2-0            trace.to_cartservice2-0.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
16        checkoutservice2-0        trace.to_currencyservice2-0.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
17        checkoutservice2-0           trace.to_emailservice2-0.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
18        checkoutservice2-0         trace.to_paymentservice2-0.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000
19        checkoutservice2-0  trace.to_productcatalogservice2-0.error_rate     28  0.000000  0.000             28            2022-03-21 07:00:00+08:00              0.00000                0.000)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables if available
try:
    trace_thresholds, trace_window, tz
except NameError:
    trace_thresholds = pd.read_csv('trace.csv')
    trace_window = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')
    trace_window['ts'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Candidate components and filter pattern
candidates = ['currencyservice2-0','productcatalogservice2-0','checkoutservice2-0','frontend2-0']
pattern = 'duration|error_rate'

# Filter global thresholds for relevant components and trace_name pattern (do NOT recompute thresholds)
thresh_sub = trace_thresholds[
    trace_thresholds['cmdb_id'].isin(candidates) &
    trace_thresholds['trace_name'].str.contains(pattern, case=False, na=False)
].copy()

# Filter window rows similarly
window_sub = trace_window[
    trace_window['cmdb_id'].isin(candidates) &
    trace_window['trace_name'].str.contains(pattern, case=False, na=False)
].copy()

# Join thresholds to window rows on (cmdb_id, trace_name)
joined = window_sub.merge(thresh_sub, on=['cmdb_id','trace_name'], how='left', suffixes=('','_global'))

# Per-row anomaly flag using precomputed global p95/p5
joined['is_anomaly'] = (
    (joined['p95'].notnull() & (joined['value'] >= joined['p95'])) |
    (joined['p5'].notnull()  & (joined['value'] <= joined['p5']))
)

# Aggregate per (cmdb_id, trace_name)
grouped = joined.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count = ('value','count'),
    p95 = ('p95','first'),
    p5  = ('p5','first'),
    anomaly_count = ('is_anomaly','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min')
)

# earliest anomaly timestamp per group (if any)
earliest_anom = (
    joined[joined['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts':'earliest_anomaly_timestamp_in_window'})
)

result = grouped.merge(earliest_anom, on=['cmdb_id','trace_name'], how='left')

# Ensure timezone for earliest_anomaly_timestamp_in_window (NaT stays)
result['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(result['earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)

# Filter to rows where (global p95>0 OR global p5>0) AND anomaly_count>0
mask_nonzero_anom = ((result['p95'].fillna(0) > 0) | (result['p5'].fillna(0) > 0)) & (result['anomaly_count'] > 0)
filtered = result[mask_nonzero_anom].copy()

if not filtered.empty:
    # Sort by earliest anomaly timestamp ascending (NaT last), then anomaly_count desc
    filtered_sorted = filtered.sort_values(['earliest_anomaly_timestamp_in_window','anomaly_count'], ascending=[True, False], na_position='last').head(20).reset_index(drop=True)
    output = filtered_sorted[[
        'cmdb_id','trace_name','count','p95','p5','anomaly_count','earliest_anomaly_timestamp_in_window','max_value_in_window','min_value_in_window'
    ]].head(20)
else:
    # Fallback: compute deviation and return top 20 by deviation desc (tie-breaker anomaly_count desc)
    def compute_dev(r):
        devs = []
        if pd.notnull(r['p95']):
            devs.append(abs(r['max_value_in_window'] - r['p95']))
        if pd.notnull(r['p5']):
            devs.append(abs(r['min_value_in_window'] - r['p5']))
        return float(max(devs)) if devs else 0.0
    result['deviation'] = result.apply(compute_dev, axis=1)
    fallback = result.sort_values(['deviation','anomaly_count'], ascending=[False, False]).head(20).reset_index(drop=True)
    output = fallback[[
        'cmdb_id','trace_name','count','p95','p5','anomaly_count','earliest_anomaly_timestamp_in_window','max_value_in_window','min_value_in_window'
    ]].head(20)

# Expose intermediates and compact final output
thresh_sub_shape = thresh_sub.shape
window_sub_shape = window_sub.shape
thresh_sub_shape, window_sub_shape, output

```
Out[8]:
```
Summary (plain English):

- What I looked for: trace duration and error_rate series for currencyservice2-0, productcatalogservice2-0, checkoutservice2-0, and frontend2-0, keeping only rows where global p95 or p5 > 0 and anomaly_count > 0 in the incident window.

- Main outcome: Several duration and one error_rate series for these components show true (non-zero-threshold) anomalies, with earliest anomalies concentrated at the start of the window (mostly 07:00–07:01+08). Frontend2-0 and checkoutservice2-0 appear repeatedly and should be prioritized.

- Representative anomalous entries (earliest-first, key fields):
  - frontend2-0 — trace.to_cartservice2-0.duration_p95: p95=0.001, anomaly_count=30, earliest=2022-03-21 07:00:00+08.
  - checkoutservice2-0 — trace.to_cartservice2-0.duration_mean: p95=0.000167, anomaly_count=28, earliest=07:00:00+08.
  - checkoutservice2-0 — trace.to_cartservice2-0.duration_p95: p95=0.000750, anomaly_count=28, earliest=07:00:00+08.
  - frontend2-0 — trace.self.error_rate: p95=0.059366, anomaly_count=28, earliest=07:00:00+08 (max ~0.0694).
  - frontend2-0 — trace.to_recommendationservice2-0.duration_p95: p95=0.003391, anomaly_count=4, earliest=07:00:00+08.
  - checkoutservice2-0 — several from_frontend2-0 and from_root duration metrics with p95 > 0 and anomaly_count = 4, earliest ≈07:01:00+08 (e.g., from_frontend2-0.duration_p95 p95≈0.0959).

- Interpretation:
  - The earliest non-zero-threshold anomalies cluster at the very start of the window (07:00–07:01), indicating an immediate change in latency/error behavior for frontend2 and downstream services.
  - Frontend2-0 shows both elevated error_rate (non-zero p95) and multiple duration anomalies toward cart/recommendation/productcatalog paths.
  - Checkoutservice2-0 shows multiple duration anomalies (from_frontend2-0, to_cartservice, to_currencyservice, etc.) with consistent early timestamps — indicates that checkoutservice2-0 experienced latency shifts very early in the window.
  - Some anomalies cover all samples in the window (anomaly_count equals count), suggesting the metric value exceeded thresholds consistently during the window rather than brief spikes.

- Recommended next steps:
  1. Prioritize investigation of frontend2-0 and checkoutservice2-0 around 07:00–07:02. Inspect service logs and traces for errors, timeouts, or dependency slowdowns at those timestamps.
  2. Correlate these trace-duration anomalies with frontend error logs and relevant metrics (CPU throttling, I/O, network) to determine whether the cause is resource contention, a traffic surge, or an upstream dependency problem.
  3. Focus on pathways flagged above: frontend2 → cartservice, currencyservice, productcatalogservice, and recommendationservice, since these show the largest/earliest deviations.

The original code execution output of IPython Kernel is also provided below for reference:

((66, 5), (1908, 5),                cmdb_id                                       trace_name  count       p95        p5  anomaly_count earliest_anomaly_timestamp_in_window  max_value_in_window  min_value_in_window
0          frontend2-0             trace.to_cartservice2-0.duration_p95     30  0.001000  0.001000             30            2022-03-21 07:00:00+08:00             0.001000             0.001000
1   checkoutservice2-0            trace.to_cartservice2-0.duration_mean     28  0.000167  0.000000             28            2022-03-21 07:00:00+08:00             0.000250             0.000000
2   checkoutservice2-0             trace.to_cartservice2-0.duration_p95     28  0.000750  0.000000             28            2022-03-21 07:00:00+08:00             0.000850             0.000000
3          frontend2-0                            trace.self.error_rate     30  0.059366  0.000000             28            2022-03-21 07:00:00+08:00             0.069401             0.000000
4          frontend2-0  trace.to_recommendationservice2-0.duration_mean     30  0.002755  0.002636              4            2022-03-21 07:00:00+08:00             0.002775             0.002615
5          frontend2-0   trace.to_recommendationservice2-0.duration_p95     30  0.003391  0.002977              4            2022-03-21 07:00:00+08:00             0.003420             0.002902
6          frontend2-0              trace.to_adservice2-0.duration_mean     30  0.000025  0.000022             11            2022-03-21 07:01:00+08:00             0.000026             0.000021
7   checkoutservice2-0         trace.to_shippingservice2-0.duration_p95     28  0.000076  0.000058              6            2022-03-21 07:01:00+08:00             0.000089             0.000058
8   checkoutservice2-0        trace.to_shippingservice2-0.duration_mean     28  0.000059  0.000053              5            2022-03-21 07:01:00+08:00             0.000064             0.000052
9   checkoutservice2-0             trace.from_frontend2-0.duration_mean     28  0.085707  0.040225              4            2022-03-21 07:01:00+08:00             0.089880             0.038737
10  checkoutservice2-0              trace.from_frontend2-0.duration_p95     28  0.095899  0.040957              4            2022-03-21 07:01:00+08:00             0.104527             0.038805
11  checkoutservice2-0                    trace.from_root.duration_mean     28  0.017412  0.002819              4            2022-03-21 07:01:00+08:00             0.019083             0.002624
12  checkoutservice2-0                     trace.from_root.duration_p95     28  0.052153  0.003476              4            2022-03-21 07:01:00+08:00             0.054686             0.003049
13  checkoutservice2-0                         trace.self.duration_mean     28  0.004544  0.003795              4            2022-03-21 07:01:00+08:00             0.006255             0.003754
14  checkoutservice2-0                          trace.self.duration_p95     28  0.008255  0.006763              4            2022-03-21 07:01:00+08:00             0.008599             0.006234
15  checkoutservice2-0         trace.to_currencyservice2-0.duration_p95     28  0.009623  0.000106              4            2022-03-21 07:01:00+08:00             0.010303             0.000104
16  currencyservice2-0       trace.from_checkoutservice2-0.duration_p95     28  0.009623  0.000106              4            2022-03-21 07:01:00+08:00             0.010303             0.000104
17         frontend2-0            trace.to_cartservice2-0.duration_mean     30  0.000377  0.000189              4            2022-03-21 07:01:00+08:00             0.000429             0.000185
18         frontend2-0        trace.to_checkoutservice2-0.duration_mean     28  0.085707  0.040225              4            2022-03-21 07:01:00+08:00             0.089880             0.038737
19         frontend2-0         trace.to_checkoutservice2-0.duration_p95     28  0.095899  0.040957              4            2022-03-21 07:01:00+08:00             0.104527             0.038805)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables if present; otherwise reconstruct from trace_thresholds and trace_window
tz = pytz.timezone('Asia/Shanghai')

try:
    per_trace_analysis  # try to use existing variable
except NameError:
    # Build per-(cmdb_id, trace_name) analysis from trace_thresholds and trace_window
    try:
        trace_thresholds, trace_window
    except NameError:
        # load fallback (should not be necessary in this kernel)
        trace_thresholds = pd.read_csv('trace.csv')
        trace_window = pd.read_csv('trace.csv')
        trace_window['ts'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

    # Join thresholds to window rows
    twj = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left', suffixes=('','_global'))

    # Per-row anomaly flag using global thresholds
    twj['is_anomaly'] = (
        twj['p95'].notnull() & (twj['value'] >= twj['p95'])
    ) | (
        twj['p5'].notnull() & (twj['value'] <= twj['p5'])
    )

    # Aggregate per trace series
    per_trace_analysis = twj.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        count = ('value','count'),
        p95 = ('p95','first'),
        p5  = ('p5','first'),
        anomaly_count = ('is_anomaly','sum'),
        max_value_in_window = ('value','max'),
        min_value_in_window = ('value','min'),
    )

    # earliest anomaly timestamp
    earliest = (
        twj[twj['is_anomaly']]
        .groupby(['cmdb_id','trace_name'], as_index=False)['ts']
        .min()
        .rename(columns={'ts':'earliest_anomaly_timestamp_in_window'})
    )
    per_trace_analysis = per_trace_analysis.merge(earliest, on=['cmdb_id','trace_name'], how='left')
    per_trace_analysis['earliest_anomaly_timestamp_in_window'] = pd.to_datetime(per_trace_analysis['earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)

    # compute deviation
    def _dev(r):
        parts = []
        if pd.notnull(r['p95']):
            parts.append(abs(r['max_value_in_window'] - r['p95']))
        if pd.notnull(r['p5']):
            parts.append(abs(r['min_value_in_window'] - r['p5']))
        return float(max(parts)) if parts else 0.0
    per_trace_analysis['deviation'] = per_trace_analysis.apply(_dev, axis=1)

# Now compute component-level severity summary
# (1) earliest_anomaly = min earliest_anomaly_timestamp_in_window across its trace_names
# (2) max_deviation = max deviation across its trace_names
# (3) total_anomaly_count = sum of anomaly_count across its trace_names
# (4) affected_trace_kpis = count of trace_names with anomaly_count>0 and (p95>0 OR p5>0)

# Ensure required columns exist
cols_needed = ['cmdb_id','trace_name','p95','p5','anomaly_count','earliest_anomaly_timestamp_in_window','deviation']
for c in cols_needed:
    if c not in per_trace_analysis.columns:
        raise RuntimeError(f"Missing expected column in per_trace_analysis: {c}")

# Compute affected_trace_kpis per trace (boolean)
per_trace_analysis['affected_flag'] = ((per_trace_analysis['anomaly_count'] > 0) & ((per_trace_analysis['p95'].fillna(0) > 0) | (per_trace_analysis['p5'].fillna(0) > 0))).astype(int)

# Group by component
component_severity = per_trace_analysis.groupby('cmdb_id', as_index=False).agg(
    earliest_anomaly = ('earliest_anomaly_timestamp_in_window', lambda s: s.min() if s.notna().any() else pd.NaT),
    max_deviation = ('deviation','max'),
    total_anomaly_count = ('anomaly_count','sum'),
    affected_trace_kpis = ('affected_flag','sum')
)

# Ensure timezone-awareness for earliest_anomaly
component_severity['earliest_anomaly'] = pd.to_datetime(component_severity['earliest_anomaly']).dt.tz_convert(tz)

# Sort by max_deviation desc, then earliest_anomaly asc (NaT last)
component_severity_sorted = component_severity.sort_values(['max_deviation','earliest_anomaly'], ascending=[False, True], na_position='last').reset_index(drop=True)

# Compact display: select requested columns and limit to top 10
component_severity_display = component_severity_sorted[['cmdb_id','affected_trace_kpis','total_anomaly_count','max_deviation','earliest_anomaly']].head(10)

component_severity_display

```
Out[9]:
```
summary = (
    "Component-level severity summary (top 10 by deviation):\n\n"
    "1) currencyservice2-0 — affected_trace_kpis=6, total_anomaly_count=85, max_deviation=44.15, earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "2) frontend2-0         — affected_trace_kpis=28, total_anomaly_count=417, max_deviation=44.15, earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "3) checkoutservice2-0  — affected_trace_kpis=27, total_anomaly_count=463, max_deviation=15.00, earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "4) productcatalogservice2-0 — affected_trace_kpis=6, total_anomaly_count=85, max_deviation=14.10, earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "5) frontend-1          — affected_trace_kpis=68, total_anomaly_count=1125, max_deviation=11.95, earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "6) cartservice2-0      — affected_trace_kpis=8, total_anomaly_count=253, max_deviation=11.00, earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "7) frontend-0          — affected_trace_kpis=70, total_anomaly_count=1183, max_deviation=6.25,  earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "8) adservice2-0        — affected_trace_kpis=3, total_anomaly_count=51,  max_deviation=5.45,  earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "9) frontend-2          — affected_trace_kpis=64, total_anomaly_count=1112, max_deviation=4.45,  earliest_anomaly=2022-03-21 07:00:00+08:00\n"
    "10) currencyservice-0  — affected_trace_kpis=18, total_anomaly_count=220, max_deviation=4.35, earliest_anomaly=2022-03-21 07:00:00+08:00\n\n"
    "Key takeaways:\n"
    "- frontend2-0 and checkoutservice2-0 have high deviation and substantial anomaly counts; they should be top priorities for investigation.\n"
    "- Multiple frontend instances (frontend-0/1/2 and frontend2-0) show many affected trace KPIs and very high total anomaly counts, indicating widespread trace-level impact starting around 07:00+08:00.\n"
    "- Many earliest anomalies cluster at ~07:00, suggesting the incident started immediately at the window start.\n\n"
    "Recommended next steps:\n"
    "1) Prioritize frontend2-0 and checkoutservice2-0 for immediate inspection (logs, traces, and related metrics) at ~07:00–07:10.\n"
    "2) Correlate these trace-level findings with frontend error logs and node/pod metrics (CPU, throttling, I/O, network) to identify resource vs. application causes.\n"
    "3) Focus on the specific downstream paths flagged (currencyservice2, cartservice, productcatalog, recommendation) for root-cause analysis.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  affected_trace_kpis  total_anomaly_count  max_deviation          earliest_anomaly
0        currencyservice2-0                    6                   85          44.15 2022-03-21 07:00:00+08:00
1               frontend2-0                   28                  417          44.15 2022-03-21 07:00:00+08:00
2        checkoutservice2-0                   27                  463          15.00 2022-03-21 07:00:00+08:00
3  productcatalogservice2-0                    6                   85          14.10 2022-03-21 07:00:00+08:00
4                frontend-1                   68                 1125          11.95 2022-03-21 07:00:00+08:00
5            cartservice2-0                    8                  253          11.00 2022-03-21 07:00:00+08:00
6                frontend-0                   70                 1183           6.25 2022-03-21 07:00:00+08:00
7              adservice2-0                    3                   51           5.45 2022-03-21 07:00:00+08:00
8                frontend-2                   64                 1112           4.45 2022-03-21 07:00:00+08:00
9         currencyservice-0                   18                  220           4.35 2022-03-21 07:00:00+08:00```
```

In [ ]:
# In[10]:

import re
import pandas as pd
import pytz
from datetime import datetime

# Reuse kernel variables
try:
    trace_window, trace_thresholds, tz
except NameError:
    trace_window = pd.read_csv('trace.csv')
    trace_thresholds = pd.read_csv('trace.csv')
    tz = pytz.timezone('Asia/Shanghai')
    trace_window['ts'] = pd.to_datetime(trace_window['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

frontends = ['frontend-0','frontend-1','frontend-2','frontend2-0']
pattern_to = r'^trace\.to_'
pattern_keep = r'duration|error_rate'

# 1) Find frontend "trace.to_" rows in window, join thresholds, compute per-(cmdb_id, trace_name) anomaly aggregates
fw = trace_window[
    trace_window['cmdb_id'].isin(frontends) &
    trace_window['trace_name'].str.match(pattern_to) &
    trace_window['trace_name'].str.contains(pattern_keep, case=False, na=False)
].copy()

# Join global thresholds (precomputed) to window rows
fwj = fw.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left', suffixes=('','_global'))

# Per-row anomaly flag using global p95/p5
fwj['is_anomaly'] = (
    (fwj['p95'].notnull() & (fwj['value'] >= fwj['p95'])) |
    (fwj['p5'].notnull()  & (fwj['value'] <= fwj['p5']))
)

# Aggregate frontend-side per (frontend_cmdb, trace_name)
frontend_agg = (
    fwj.groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        count=('value','count'),
        frontend_p95=('p95','first'),
        frontend_p5=('p5','first'),
        frontend_anomaly_count=('is_anomaly','sum'),
    )
)

# earliest anomaly timestamp per frontend trace
fa_earliest = (
    fwj[fwj['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts':'frontend_earliest_anomaly_timestamp_in_window'})
)
frontend_agg = frontend_agg.merge(fa_earliest, on=['cmdb_id','trace_name'], how='left')
frontend_agg['frontend_earliest_anomaly_timestamp_in_window'] = pd.to_datetime(frontend_agg['frontend_earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)

# Keep only those frontend trace entries where (p95>0 OR p5>0) AND anomaly_count>0
frontend_candidates = frontend_agg[
    (
        (frontend_agg['frontend_p95'].fillna(0) > 0) |
        (frontend_agg['frontend_p5'].fillna(0) > 0)
    ) &
    (frontend_agg['frontend_anomaly_count'] > 0)
].copy()

# Extract target cmdb_id from frontend trace_name: 'trace.to_<target>.' -> capture <target>
def extract_target(trace_name):
    m = re.search(r'^trace\.to_([^\.]+)', trace_name)
    return m.group(1) if m else None

frontend_candidates['target_cmdb'] = frontend_candidates['trace_name'].apply(extract_target)

# 2) Prepare target-side aggregates: for all possible target cmdb_ids, find trace.from_<frontend> rows in window
# First, build target aggregate table for all trace series in window joined with thresholds
twj = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left', suffixes=('','_global'))
twj['is_anomaly'] = (
    (twj['p95'].notnull() & (twj['value'] >= twj['p95'])) |
    (twj['p5'].notnull()  & (twj['value'] <= twj['p5']))
)
target_agg_all = (
    twj.groupby(['cmdb_id','trace_name'], as_index=False)
    .agg(
        count=('value','count'),
        target_p95=('p95','first'),
        target_p5=('p5','first'),
        target_anomaly_count=('is_anomaly','sum'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
    )
)
# earliest anomaly ts for targets
ta_earliest = (
    twj[twj['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], as_index=False)['ts']
    .min()
    .rename(columns={'ts':'target_earliest_anomaly_timestamp_in_window'})
)
target_agg_all = target_agg_all.merge(ta_earliest, on=['cmdb_id','trace_name'], how='left')
target_agg_all['target_earliest_anomaly_timestamp_in_window'] = pd.to_datetime(target_agg_all['target_earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)

# 3) For each frontend candidate, find matching target-side rows of pattern 'trace.from_<frontend_cmdb>'
rows = []
for _, fr in frontend_candidates.iterrows():
    f_cmdb = fr['cmdb_id']
    f_trace = fr['trace_name']
    f_earliest = fr['frontend_earliest_anomaly_timestamp_in_window']
    tgt = fr['target_cmdb']
    if pd.isna(tgt):
        # no parsable target; emit single row with nulls
        rows.append({
            'frontend_cmdb': f_cmdb,
            'target_cmdb': None,
            'frontend_trace_name': f_trace,
            'frontend_earliest_anomaly_timestamp_in_window': f_earliest,
            'target_trace_name': None,
            'target_earliest_anomaly_timestamp_in_window': pd.NaT
        })
        continue

    # Candidate target rows for this target cmdb that represent calls FROM this frontend
    pattern_from = rf'^trace\.from_{re.escape(f_cmdb)}.*'
    candidates_target = target_agg_all[
        (target_agg_all['cmdb_id'] == tgt) &
        (target_agg_all['trace_name'].str.match(pattern_from)) &
        (target_agg_all['trace_name'].str.contains(pattern_keep, case=False, na=False)) &
        (
            (target_agg_all['target_p95'].fillna(0) > 0) |
            (target_agg_all['target_p5'].fillna(0) > 0)
        ) &
        (target_agg_all['target_anomaly_count'] > 0)
    ].copy()

    if candidates_target.empty:
        rows.append({
            'frontend_cmdb': f_cmdb,
            'target_cmdb': tgt,
            'frontend_trace_name': f_trace,
            'frontend_earliest_anomaly_timestamp_in_window': f_earliest,
            'target_trace_name': None,
            'target_earliest_anomaly_timestamp_in_window': pd.NaT
        })
    else:
        # produce one output row per matching target trace_name
        for _, tr in candidates_target.iterrows():
            rows.append({
                'frontend_cmdb': f_cmdb,
                'target_cmdb': tgt,
                'frontend_trace_name': f_trace,
                'frontend_earliest_anomaly_timestamp_in_window': f_earliest,
                'target_trace_name': tr['trace_name'],
                'target_earliest_anomaly_timestamp_in_window': tr['target_earliest_anomaly_timestamp_in_window']
            })

# Build final DataFrame, sort and limit
result_df = pd.DataFrame(rows)
# Ensure datetime type for sorting
result_df['frontend_earliest_anomaly_timestamp_in_window'] = pd.to_datetime(result_df['frontend_earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)
result_df['target_earliest_anomaly_timestamp_in_window'] = pd.to_datetime(result_df['target_earliest_anomaly_timestamp_in_window']).dt.tz_convert(tz)

result_df_sorted = result_df.sort_values(
    ['frontend_earliest_anomaly_timestamp_in_window','target_earliest_anomaly_timestamp_in_window'],
    ascending=[True, True],
    na_position='last'
).head(50).reset_index(drop=True)

# Compact output columns
result_df_sorted[['frontend_cmdb','target_cmdb','frontend_trace_name','frontend_earliest_anomaly_timestamp_in_window','target_trace_name','target_earliest_anomaly_timestamp_in_window']]

```
Out[10]:
```
```python
# Summarize frontend->target trace matching findings using kernel-stored result_df_sorted
try:
    df = result_df_sorted
except NameError:
    df = None

if df is None or df.empty:
    summary = "No frontend->target trace matches found in the kernel-stored results."
else:
    total_pairs = df.shape[0]
    # show up to 6 example mappings
    examples = []
    for _, r in df.head(6).iterrows():
        fcm = r['frontend_cmdb']
        tcm = r['target_cmdb'] if pd.notna(r['target_cmdb']) else 'N/A'
        ftrace = r['frontend_trace_name']
        ttrace = r['target_trace_name'] if pd.notna(r['target_trace_name']) else 'N/A'
        fts = r['frontend_earliest_anomaly_timestamp_in_window']
        tts = r['target_earliest_anomaly_timestamp_in_window']
        examples.append(f"{fcm} -> {tcm}: frontend_trace='{ftrace}' (earliest={fts}), target_trace='{ttrace}' (earliest={tts})")
    examples_text = "\n  ".join(examples)

    # compute counts of pairs with/without matching target-side row
    matched = df['target_trace_name'].notna().sum()
    unmatched = total_pairs - matched

    # earliest overall frontend anomaly
    earliest_frontend = df['frontend_earliest_anomaly_timestamp_in_window'].min()

    summary = (
        f"Frontend→Target trace pairing summary ({total_pairs} rows examined):\n\n"
        f"- Most frontend-to-target trace anomalies have matching target-side 'from_<frontend>' traces with anomalies occurring at the same minute.\n"
        f"- Total pairs with a matching target-side anomalous trace: {matched}; without match: {unmatched}.\n"
        f"- Earliest frontend-side anomaly observed: {earliest_frontend}.\n\n"
        f"Example mappings (up to 6):\n  {examples_text}\n\n"
        "Interpretation & recommended next steps:\n"
        "1) Many downstream services (adservice, currencyservice, productcatalogservice, cartservice, etc.) show corresponding anomalous traces immediately when frontends report anomalies — this suggests the issue propagated from frontends to downstream dependencies or a shared upstream problem.\n"
        "2) Prioritize investigation of frontend->target pairs listed above (check target service logs and 'from_<frontend>' spans around the reported earliest timestamps, notably ~07:00–07:02).\n"
        "3) If a frontend->target pair has no matching target-side anomalous trace (unmatched rows), consider instrumentation gaps or timing mismatches; verify traces exist on the target and that naming conventions align.\n"
        "4) Correlate these findings with frontend error logs and node/pod metrics at the same timestamps to determine whether root cause is resource-related, network-related, or an application-level failure."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

frontend_cmdb        target_cmdb                       frontend_trace_name frontend_earliest_anomaly_timestamp_in_window                    target_trace_name target_earliest_anomaly_timestamp_in_window
0     frontend-0        adservice-0        trace.to_adservice-0.duration_mean                     2022-03-21 07:00:00+08:00  trace.from_frontend-0.duration_mean                   2022-03-21 07:00:00+08:00
1     frontend-0        adservice-2        trace.to_adservice-2.duration_mean                     2022-03-21 07:00:00+08:00  trace.from_frontend-0.duration_mean                   2022-03-21 07:00:00+08:00
2     frontend-0        adservice-2        trace.to_adservice-2.duration_mean                     2022-03-21 07:00:00+08:00   trace.from_frontend-0.duration_p95                   2022-03-21 07:00:00+08:00
3     frontend-0        adservice-2         trace.to_adservice-2.duration_p95                     2022-03-21 07:00:00+08:00  trace.from_frontend-0.duration_mean                   2022-03-21 07:00:00+08:00
4     frontend-0        adservice-2         trace.to_adservice-2.duration_p95                     2022-03-21 07:00:00+08:00   trace.from_frontend-0.duration_p95                   2022-03-21 07:00:00+08:00
..           ...                ...                                       ...                                           ...                                  ...                                         ...
45    frontend-2        adservice-1         trace.to_adservice-1.duration_p95                     2022-03-21 07:00:00+08:00   trace.from_frontend-2.duration_p95                   2022-03-21 07:00:00+08:00
46    frontend-2  currencyservice-0   trace.to_currencyservice-0.duration_p95                     2022-03-21 07:00:00+08:00   trace.from_frontend-2.duration_p95                   2022-03-21 07:00:00+08:00
47    frontend-2  currencyservice-1  trace.to_currencyservice-1.duration_mean                     2022-03-21 07:00:00+08:00  trace.from_frontend-2.duration_mean                   2022-03-21 07:00:00+08:00
48    frontend-2  currencyservice-1  trace.to_currencyservice-1.duration_mean                     2022-03-21 07:00:00+08:00   trace.from_frontend-2.duration_p95                   2022-03-21 07:00:00+08:00
49    frontend-2  currencyservice-1   trace.to_currencyservice-1.duration_p95                     2022-03-21 07:00:00+08:00  trace.from_frontend-2.duration_mean                   2022-03-21 07:00:00+08:00

[50 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz

# Reuse existing result_df_sorted from kernel
tz = pytz.timezone('Asia/Shanghai')

# Ensure result_df_sorted exists
try:
    result_df_sorted
except NameError:
    raise RuntimeError("result_df_sorted not found in kernel. Please run the previous mapping step first.")

# Ensure datetime columns are timezone-aware
for col in ['frontend_earliest_anomaly_timestamp_in_window','target_earliest_anomaly_timestamp_in_window']:
    if col in result_df_sorted.columns:
        result_df_sorted[col] = pd.to_datetime(result_df_sorted[col]).dt.tz_convert(tz)

# 1) Filter rows where target_trace_name is not null and target_earliest <= frontend_earliest
mask = result_df_sorted['target_trace_name'].notna() & \
       result_df_sorted['target_earliest_anomaly_timestamp_in_window'].notna() & \
       (result_df_sorted['target_earliest_anomaly_timestamp_in_window'] <= result_df_sorted['frontend_earliest_anomaly_timestamp_in_window'])

filtered = result_df_sorted[mask].copy()

# 2) Aggregate by target_cmdb
agg = (
    filtered
    .groupby('target_cmdb', as_index=False)
    .agg(
        count_pairs_where_target_before_or_equal = ('target_cmdb','count'),
        earliest_target_anomaly_timestamp = ('target_earliest_anomaly_timestamp_in_window','min')
    )
)

# Sort and limit to top 10 as requested
agg_sorted = agg.sort_values(['count_pairs_where_target_before_or_equal','earliest_target_anomaly_timestamp'],
                            ascending=[False, True], na_position='last').head(10).reset_index(drop=True)

# 3) Sample up to 10 frontend->target rows from filtered set, sorted by frontend earliest asc
sample_rows = (
    filtered
    .sort_values('frontend_earliest_anomaly_timestamp_in_window', ascending=True)
    .loc[:, ['frontend_cmdb','target_cmdb','frontend_trace_name','frontend_earliest_anomaly_timestamp_in_window','target_trace_name','target_earliest_anomaly_timestamp_in_window']]
    .head(10)
    .reset_index(drop=True)
)

# Compact outputs
agg_sorted, sample_rows

```
Out[11]:
```
```python
summary = (
    "Summary of frontend→target timing analysis (rows where target anomaly occurred at or before frontend):\n\n"
    "Top targets with the most frontend→target pairs where the target anomaly precedes or ties:\n"
    " - cartservice-0: 8 pairs (earliest target anomaly 2022-03-21 07:00:00+08)\n"
    " - cartservice-1: 8 pairs (earliest 2022-03-21 07:00:00+08)\n"
    " - adservice-1: 5 pairs (earliest 2022-03-21 07:00:00+08)\n"
    " - shippingservice-0: 5 pairs (earliest 2022-03-21 07:00:00+08)\n"
    " - adservice-2, cartservice-2, checkoutservice-0, currencyservice-2: 4 pairs each (earliest 07:00+08)\n\n"
    "Representative frontend→target examples (earliest-first):\n"
    " - frontend-0 -> adservice-0: frontend trace 'trace.to_adservice-0.duration_mean' (07:00), target trace 'trace.from_frontend-0.duration_mean' (07:00)\n"
    " - frontend-1 -> cartservice-0: frontend 'trace.to_cartservice-0.duration_mean' (07:00), target 'trace.from_frontend-1.duration_mean' (07:00)\n"
    " - frontend-1 -> cartservice-1: frontend 'trace.to_cartservice-1.duration_p95' (07:00), target 'trace.from_frontend-1.duration_p95' (07:00)\n\n"
    "Interpretation and recommended next steps:\n"
    "1) Multiple target services (notably cartservice instances and several adservice/shippingservice instances) show anomalies at or before the frontend anomalies, suggesting the problem may originate at those targets or a shared upstream service.\n"
    "2) Prioritize investigating target-side logs and traces for cartservice-0 and cartservice-1 first (they have the most preceding pairs and earliest anomalies at ~07:00).\n"
    "3) Correlate target logs/metrics (latency, errors, CPU, network) at the reported timestamps with frontend errors to confirm causality. If targets show no issues, check shared dependencies or network/router components.\n"
    "4) For frontend→target pairs without a matching target-side anomaly, validate instrumentation and trace naming to rule out missing spans.\n\n"
    "If you want, I can (a) show full matched rows for a specific target, (b) extract target log samples at the earliest timestamps, or (c) correlate these times with node/pod metrics."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         target_cmdb  count_pairs_where_target_before_or_equal earliest_target_anomaly_timestamp
0      cartservice-0                                         8         2022-03-21 07:00:00+08:00
1      cartservice-1                                         8         2022-03-21 07:00:00+08:00
2        adservice-1                                         5         2022-03-21 07:00:00+08:00
3  shippingservice-0                                         5         2022-03-21 07:00:00+08:00
4        adservice-2                                         4         2022-03-21 07:00:00+08:00
5      cartservice-2                                         4         2022-03-21 07:00:00+08:00
6  checkoutservice-0                                         4         2022-03-21 07:00:00+08:00
7  currencyservice-2                                         4         2022-03-21 07:00:00+08:00
8  currencyservice-1                                         3         2022-03-21 07:00:00+08:00
9  currencyservice-0                                         2         2022-03-21 07:00:00+08:00,   frontend_cmdb    target_cmdb                   frontend_trace_name frontend_earliest_anomaly_timestamp_in_window                    target_trace_name target_earliest_anomaly_timestamp_in_window
0    frontend-0    adservice-0    trace.to_adservice-0.duration_mean                     2022-03-21 07:00:00+08:00  trace.from_frontend-0.duration_mean                   2022-03-21 07:00:00+08:00
1    frontend-1    adservice-1     trace.to_adservice-1.duration_p95                     2022-03-21 07:00:00+08:00   trace.from_frontend-1.duration_p95                   2022-03-21 07:00:00+08:00
2    frontend-1  cartservice-0  trace.to_cartservice-0.duration_mean                     2022-03-21 07:00:00+08:00  trace.from_frontend-1.duration_mean                   2022-03-21 07:00:00+08:00
3    frontend-1  cartservice-0  trace.to_cartservice-0.duration_mean                     2022-03-21 07:00:00+08:00   trace.from_frontend-1.duration_p95                   2022-03-21 07:00:00+08:00
4    frontend-1  cartservice-0   trace.to_cartservice-0.duration_p95                     2022-03-21 07:00:00+08:00  trace.from_frontend-1.duration_mean                   2022-03-21 07:00:00+08:00
5    frontend-1  cartservice-0   trace.to_cartservice-0.duration_p95                     2022-03-21 07:00:00+08:00   trace.from_frontend-1.duration_p95                   2022-03-21 07:00:00+08:00
6    frontend-1  cartservice-1  trace.to_cartservice-1.duration_mean                     2022-03-21 07:00:00+08:00  trace.from_frontend-1.duration_mean                   2022-03-21 07:00:00+08:00
7    frontend-1  cartservice-1  trace.to_cartservice-1.duration_mean                     2022-03-21 07:00:00+08:00   trace.from_frontend-1.duration_p95                   2022-03-21 07:00:00+08:00
8    frontend-1  cartservice-1   trace.to_cartservice-1.duration_p95                     2022-03-21 07:00:00+08:00  trace.from_frontend-1.duration_mean                   2022-03-21 07:00:00+08:00
9    frontend-1  cartservice-1   trace.to_cartservice-1.duration_p95                     2022-03-21 07:00:00+08:00   trace.from_frontend-1.duration_p95                   2022-03-21 07:00:00+08:00)```
```